In [ ]:
import sys

ENVIRONMENT = sys.argv[1]

project_name = "FEATURE_STORE_DEPLOYMENT_LOGS_ARGS_" + ENVIRONMENT

if ENVIRONMENT in ("DEV", "QA"):

    FS_DB = "FEATURE_STORE_" + ENVIRONMENT

elif ENVIRONMENT == "PRD":

    FS_DB = "FEATURE_STORE"

else:

    FS_DB = "UNKNOWN"

import uuid, json

from snowflake.snowpark import Session

run_id = str(uuid.uuid4())

session = Session.builder.getOrCreate()

session.sql(f"""CREATE SCHEMA IF NOT EXISTS {FS_DB}.NOTEBOOKS_CICD""").collect()

session.sql(f""" CREATE TABLE IF NOT EXISTS {FS_DB}.NOTEBOOKS_CICD.NB_RUN_LOGS (

  run_id STRING,

  ts TIMESTAMP_LTZ,

  level STRING,

  message STRING,

  context VARIANT

)""").collect()

def log(level, message, **ctx):

    session.sql(f"""

      INSERT INTO {FS_DB}.NOTEBOOKS_CICD.NB_RUN_LOGS

      (run_id, ts, level, message, context)

      SELECT ?, CURRENT_TIMESTAMP(), ?, ?, PARSE_JSON(?)

    """, params=[run_id, level, message, json.dumps(ctx or {})]).collect()

log("INFO", "START DEPLOYMENT NOTEBOOK", project = project_name)

log("INFO", "END DEPLOYMENT NOTEBOOK", project = project_name)